In [13]:
# 순위 3강 VS 나머지
import pandas as pd

# 1. 데이터 로드
df = pd.read_csv('data/KBO_Season_Master_Data.csv')

# 2. 그룹 할당 (연도별로 1~3위는 '3강', 나머지는 'Others')
df['Group'] = df['순위'].apply(lambda x: 'Top3' if x <= 3 else 'Others')

# 3. 분석할 지표 리스트
metrics = ['승률', 'WAR_total', 'ERA', 'OPS', '득점', 'F%']

# 4. 연도별/그룹별 평균 계산
yearly_comp = df.groupby(['연도', 'Group'])[metrics].mean().reset_index()

# 5. 표를 보기 좋게 변환 (Pivot)
# 연도별로 Top3와 Others의 지표를 한 줄에 보기 위해 피벗 테이블 생성
pivot_result = yearly_comp.pivot(index='연도', columns='Group')

# 6. 격차(Gap) 컬럼 계산 (Top3 - Others)
for col in metrics:
    pivot_result[(col, 'Gap')] = pivot_result[(col, 'Top3')] - pivot_result[(col, 'Others')]

# 컬럼 순서 정렬 (지표별로 Top3, Others, Gap 순서대로 나오게)
ordered_cols = []
for col in metrics:
    ordered_cols.extend([(col, 'Top3'), (col, 'Others'), (col, 'Gap')])

final_table = pivot_result[ordered_cols]

print("--- 연도별 3강 vs 나머지 팀 지표 비교 표 ---")
display(final_table.round(3)) # 소수점 3째자리까지 출력

# 1. 분석할 지표 리스트 
summary_metrics = ['승률', 'WAR_total', 'ERA', 'OPS', '득점', 'F%']

# 2. 전체 기간에 대한 그룹별 평균 계산
total_summary = df.groupby('Group')[summary_metrics].mean()

# 3. 행과 열을 전환 (T) 하여 '지표'가 행으로 오게 함
total_summary_df = total_summary.T

# 4. Gap 컬럼 계산 (Top3 - Others)
total_summary_df['Gap'] = total_summary_df['Top3'] - total_summary_df['Others']

# 5. 컬럼 순서 정렬 (Top3, Others, Gap)
total_summary_df = total_summary_df[['Top3','Others', 'Gap']]

# 6. 지표 이름 가독성 좋게 수정
total_summary_df = total_summary_df.rename(index={'승률': '승률', 'WAR_total': 'WAR', 'ERA': 'ERA', 'OPS': 'OPS', '득점': '득점', 'F%': '수비율'})

print("\n--- 2020-2025 전 기간 통합 강팀 구조 평균 요약 ---")
display(total_summary_df.round(3))


--- 연도별 3강 vs 나머지 팀 지표 비교 표 ---


승률               WAR_total                    ERA                \
Group   Top3 Others    Gap      Top3  Others     Gap   Top3 Others    Gap   
연도                                                                          
2020   0.577  0.467  0.110    55.433  33.276  22.158  4.477  4.880 -0.403   
2021   0.549  0.479  0.071    47.047  36.693  10.354  4.077  4.593 -0.516   
2022   0.602  0.457  0.145    50.727  34.766  15.961  3.663  4.233 -0.570   
2023   0.568  0.470  0.098    47.283  35.673  11.610  3.993  4.204 -0.211   
2024   0.566  0.472  0.094    48.390  35.681  12.709  4.570  5.053 -0.483   
2025   0.577  0.467  0.111    52.700  33.183  19.517  3.657  4.590 -0.933   

         OPS                     득점                       F%                
Group   Top3 Others    Gap     Top3   Others      Gap   Top3 Others    Gap  
연도                                                                          
2020   0.805  0.737  0.068  839.000  702.714  136.286  0.983  0.981  0.001  
2021   0.740  0.724  0.016  723.000  675.286   47.714  0.982  0.980  0.002  
2022   0.724  0.708  0.016  685.333  638.143   47.190  0.981  0.978  0.002  
2023   0.730  0.705  0.025  699.000  646.429   52.571  0.978  0.979 -0.001  
2024   0.794  0.762  0.032  812.000  757.714   54.286  0.979  0.979  0.000  
2025   0.735  0.724  0.012  695.333  675.286   20.048  0.982  0.979  0.003


--- 2020-2025 전 기간 통합 강팀 구조 평균 요약 ---


Group,Top3,Others,Gap
승률,0.573,0.469,0.105
WAR,50.263,34.879,15.385
ERA,4.073,4.592,-0.519
OPS,0.755,0.726,0.028
득점,742.278,682.595,59.683
수비율,0.981,0.980,0.001


In [14]:
# 승률 3강 vs 나머지
import pandas as pd

# 1. 데이터 로드
df = pd.read_csv('data/KBO_Season_Master_Data.csv')

# 2. [핵심] 승률 기준으로 그룹 재할당
# 연도별로 승률(내림차순), 순위(오름차순) 정렬 후 상위 3개 팀 추출
def classify_by_win_rate(group):
    group = group.sort_values(by=['승률', '순위'], ascending=[False, True])
    group['Group'] = ['Top3'] * 3 + ['Others'] * (len(group) - 3)
    return group

# 연도별 그룹화 후 적용
df = df.groupby('연도', group_keys=False).apply(classify_by_win_rate)

# 3. 분석 지표 설정
metrics = ['승률', 'WAR_total', 'ERA', 'OPS', '득점', 'F%']

# ---------------------------------------------------------
# [파트 1] 연도별 상세 비교 표 (기존 형식 유지)
# ---------------------------------------------------------
yearly_comp = df.groupby(['연도', 'Group'])[metrics].mean().reset_index()
pivot_result = yearly_comp.pivot(index='연도', columns='Group')

# Gap 계산 및 정렬
for col in metrics:
    pivot_result[(col, 'Gap')] = pivot_result[(col, 'Top3')] - pivot_result[(col, 'Others')]

ordered_cols = []
for col in metrics:
    ordered_cols.extend([(col, 'Top3'), (col, 'Others'), (col, 'Gap')])

final_yearly_table = pivot_result[ordered_cols]

print("--- [승률 기준] 연도별 3강 vs 나머지 팀 지표 비교 ---")
display(final_yearly_table.round(3))


# ---------------------------------------------------------
# [파트 2] 전 기간 통합 강팀 구조 평균 요약 (행: 지표, 열: 그룹)
# ---------------------------------------------------------
total_summary = df.groupby('Group')[metrics].mean().T

# Gap 계산 및 순서 정렬
total_summary['Gap'] = total_summary['Top3'] - total_summary['Others']
total_summary = total_summary[['Top3', 'Others', 'Gap']]

# 지표 이름 변경
total_summary = total_summary.rename(index={
    '승률': '승률', 
    'WAR_total': 'WAR', 
    'ERA': 'ERA', 
    'OPS': 'OPS', 
    '득점': '득점',
    'F%': '수비율'
})

print("\n--- [승률 기준] 2020-2025 전 기간 통합 강팀 구조 평균 요약 ---")
display(total_summary.round(3))

--- [승률 기준] 연도별 3강 vs 나머지 팀 지표 비교 ---


/var/folders/qv/w4b6_7w91dlfrdzfhp8mnl080000gn/T/ipykernel_5619/1348629886.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('연도', group_keys=False).apply(classify_by_win_rate)


승률               WAR_total                    ERA                \
Group   Top3 Others    Gap      Top3  Others     Gap   Top3 Others    Gap   
연도                                                                          
2020   0.577  0.467  0.110    55.433  33.276  22.158  4.477  4.880 -0.403   
2021   0.560  0.474  0.086    47.793  36.373  11.420  3.847  4.691 -0.845   
2022   0.602  0.457  0.145    50.727  34.766  15.961  3.663  4.233 -0.570   
2023   0.568  0.470  0.098    47.283  35.673  11.610  3.993  4.204 -0.211   
2024   0.566  0.472  0.094    48.390  35.681  12.709  4.570  5.053 -0.483   
2025   0.577  0.467  0.111    52.700  33.183  19.517  3.657  4.590 -0.933   

         OPS                     득점                       F%                
Group   Top3 Others    Gap     Top3   Others      Gap   Top3 Others    Gap  
연도                                                                          
2020   0.805  0.737  0.068  839.000  702.714  136.286  0.983  0.981  0.001  
2021   0.730  0.728  0.002  695.000  687.286    7.714  0.982  0.980  0.002  
2022   0.724  0.708  0.016  685.333  638.143   47.190  0.981  0.978  0.002  
2023   0.730  0.705  0.025  699.000  646.429   52.571  0.978  0.979 -0.001  
2024   0.794  0.762  0.032  812.000  757.714   54.286  0.979  0.979  0.000  
2025   0.735  0.724  0.012  695.333  675.286   20.048  0.982  0.979  0.003


--- [승률 기준] 2020-2025 전 기간 통합 강팀 구조 평균 요약 ---


Group,Top3,Others,Gap
승률,0.575,0.468,0.107
WAR,50.388,34.825,15.563
ERA,4.034,4.609,-0.574
OPS,0.753,0.727,0.026
득점,737.611,684.595,53.016
수비율,0.981,0.980,0.001
